In [3]:
import tonic
import torch
import torch.nn as nn

import pandas as pd
import numpy as np
import random

import time
import datetime
import math

c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

SEED = 100
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(device)

cuda


# Data Loading

In [5]:
transform_dataset = tonic.transforms.Compose([
    tonic.transforms.Downsample(spatial_factor=0.5)
])

transform_train = tonic.transforms.Compose([
    tonic.transforms.RandomCrop(sensor_size=(120,90,2), target_size=(80,60)),
    tonic.transforms.RandomFlipLR(sensor_size=(80,60,2), p = 0.5),
    tonic.transforms.EventDrop(sensor_size=(80,60,2)),
    tonic.transforms.ToFrame(sensor_size=(80,60,2), n_time_bins=10)
])

transform_val = tonic.transforms.Compose([
    tonic.transforms.CenterCrop(sensor_size=(120,90,2), size=(80,60)),
    tonic.transforms.ToFrame(sensor_size=(80,60,2), n_time_bins=10)
])

In [6]:
dataset = tonic.datasets.NCALTECH101(save_to = './Data_80_60_n10' , transform=transform_dataset)

In [7]:
#changing the targets to numerical values:
classes = sorted(set(dataset.targets))
class_to_idx = {cls: i for i, cls in enumerate(classes)}
dataset.targets = [class_to_idx[target] for target in dataset.targets]

In [8]:
from torch.utils.data import random_split

train = int(len(dataset) * 0.7)
val = int(len(dataset) * 0.2)
test = len(dataset) - train - val

generator = torch.Generator().manual_seed(SEED)

train, val, test = random_split(dataset, [train, val, test], generator=generator)
len(train), len(val), len(test)

(6096, 1741, 872)

In [9]:
from torch.utils.data import DataLoader, Dataset
from tonic.cached_dataset import MemoryCachedDataset

class SafeDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        
    def __getitem__(self, idx):
        data, target = self.dataset[idx]
        while isinstance(data, np.ndarray) and data.dtype.names is not None:
            idx = np.random.randint(0, len(self.dataset))
            data, target = self.dataset[idx]
        return data, target

    def __len__(self):
        return len(self.dataset)

train_cached = tonic.MemoryCachedDataset(train, transform=transform_train)
val_cached = tonic.MemoryCachedDataset(val, transform=transform_val)
test_cached = tonic.MemoryCachedDataset(test, transform=transform_val)

train_set_safe = SafeDataset(train_cached)
val_set_safe = SafeDataset(val_cached)
test_set_safe = SafeDataset(test_cached)

batch_size = 128

train_loader = DataLoader(train_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                   shuffle = True, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                        drop_last=True, pin_memory=True)
test_loader = DataLoader(test_set_safe, batch_size=batch_size, collate_fn=tonic.collation.PadTensors(batch_first=False),
                          drop_last=True, pin_memory=True)

In [10]:
event, target = train_set_safe[200] 
print(f'Output: {event.shape} for target {target}')

Output: (10, 2, 60, 80) for target 0


# T-SEW-ResNet

In [11]:
from spikingjelly.clock_driven import layer, functional
from spikingjelly.activation_based import neuron, surrogate
import snntorch.functional as SF

In [12]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride = 1, downsample = None, connect_f = 'ADD'):
        super().__init__()
        self.connect_f = connect_f

        self.conv1 = layer.SeqToANNContainer(
            nn.Conv2d(inplanes, planes, kernel_size = 3, stride=stride, padding = 1, bias = False),
            nn.BatchNorm2d(planes))
        self.sn1 = neuron.IFNode(step_mode='m', detach_reset=True)
        
        self.conv2 = layer.SeqToANNContainer(
            nn.Conv2d(planes, planes, kernel_size = 3, stride=1, padding = 1, bias = False),
            nn.BatchNorm2d(planes))
        self.sn2 = neuron.IFNode(step_mode='m', detach_reset=True)

        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.sn1(out)

        out = self.conv2(out)
        out = self.sn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        if self.connect_f == 'ADD':
            out = out + identity
        elif self.connect_f == 'AND':
            out = out * identity
        elif self.connect_f == 'IAND':
            out = identity * (1. - out)
        else:
            raise NotImplementedError(self.connect_f)

        return out

def zero_init_blocks(net: nn.Module, connect_f: str):
    for m in net.modules():
        if isinstance(m, BasicBlock):
            nn.init.constant_(m.conv2.module[1].weight, 0)
            if connect_f == 'AND':
                nn.init.constant_(m.conv2.module[1].bias, 1)

class TSEWResNet(nn.Module):
  def __init__(self, num_classes = 101, connect_f = 'ADD', zero_init_residual=False):
    super().__init__()
    self.inplanes = 64
    self.connect_f = connect_f

    self.conv1 = layer.SeqToANNContainer(nn.Conv2d(2, self.inplanes, kernel_size = 3, stride= 1, padding = 1, bias = False))
    self.bn1 = layer.SeqToANNContainer(nn.BatchNorm2d(self.inplanes))
    self.sn1 = neuron.IFNode(step_mode='m', detach_reset=True)
    self.maxpool = layer.SeqToANNContainer(nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    self.layer1 = self._make_layer(64, 2, stride=1)
    self.layer2 = self._make_layer(128, 2, stride=2)
    self.layer3 = self._make_layer(256, 2, stride=2)

    self.avgpool = layer.SeqToANNContainer(nn.AdaptiveAvgPool2d((1, 1)))
    self.fc = nn.Linear(256 * BasicBlock.expansion, num_classes)
    self.sn_out = neuron.IFNode(step_mode='m', detach_reset=True)

    for m in self.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    if zero_init_residual:
        zero_init_blocks(self, connect_f)

  def _make_layer(self, planes, blocks, stride = 1):
    downsample = None
    if stride != 1 or self.inplanes != planes * BasicBlock.expansion:
      downsample = nn.Sequential(
        layer.SeqToANNContainer(nn.Conv2d(self.inplanes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
        nn.BatchNorm2d(planes * BasicBlock.expansion)),
        neuron.IFNode(step_mode='m', detach_reset=True)
      )

    layers = []
    layers.append(BasicBlock(self.inplanes, planes, stride, downsample, connect_f=self.connect_f))
    self.inplanes = planes * BasicBlock.expansion
    for _ in range(1, blocks):
        layers.append(BasicBlock(self.inplanes, planes, connect_f=self.connect_f))

    return nn.Sequential(*layers)
   

  def forward(self, x):

    x = self.conv1(x)
    x = self.bn1(x)
    x = self.sn1(x)
    x = self.maxpool(x)

    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)

    x = self.avgpool(x)
    x = torch.flatten(x, 2)
    x = self.fc(x)
    sn_out = self.sn_out(x)

    return sn_out

In [13]:
def _sew_resnet(**kwargs):
    return TSEWResNet(**kwargs)

# PyTorch Classification references functions

In [14]:
import utils

In [15]:
def train_one_epoch(model, criterion, optimizer, data_loader, device, epoch, print_freq, scaler=None):

    model.train()
    metric_logger = utils.MetricLogger(delimiter="  ")
    metric_logger.add_meter('lr', utils.SmoothedValue(window_size=1, fmt='{value}'))
    metric_logger.add_meter('img/s', utils.SmoothedValue(window_size=10, fmt='{value}'))

    header = 'Epoch: [{}]'.format(epoch)

    for image, target in metric_logger.log_every(data_loader, print_freq, header):
        start_time = time.time()
        image, target = image.to(device), target.to(device)
        # with torch.autograd.detect_anomaly():
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                output = model(image)
                loss = criterion(output, target)

        else:
            output = model(image)
            loss = criterion(output, target)
   
        optimizer.zero_grad()

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:
            loss.backward()
            optimizer.step()

        functional.reset_net(model)

        acc1 = SF.accuracy_temporal(output.detach(), target) * 100
        batch_size = image.shape[0]
        loss_s = loss.item()
        if math.isnan(loss_s):
            raise ValueError('loss is Nan')
        acc1_s = acc1.item()

        metric_logger.update(loss=loss_s, lr=optimizer.param_groups[0]["lr"])
        metric_logger.meters['acc1'].update(acc1_s, n=batch_size)
        metric_logger.meters['img/s'].update(batch_size / (time.time() - start_time))

    # gather the stats from all processes
    metric_logger.synchronize_between_processes()
    return metric_logger.loss.global_avg, metric_logger.acc1.global_avg



def evaluate(model, criterion, data_loader, device, print_freq=100, header='Test:'):
    model.eval()
    metric_logger = utils.MetricLogger(delimiter="  ")
    with torch.no_grad():
        for image, target in metric_logger.log_every(data_loader, print_freq, header):
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            output = model(image)
            loss = criterion(output, target)
            functional.reset_net(model)

            acc1 = SF.accuracy_temporal(output, target) * 100
            batch_size = image.shape[0]
            metric_logger.update(loss=loss.item())
            metric_logger.meters['acc1'].update(acc1.item(), n=batch_size)
    # gather the stats from all processes
    metric_logger.synchronize_between_processes()

    loss, acc1 = metric_logger.loss.global_avg, metric_logger.acc1.global_avg
    print(f' * Acc@1 = {acc1}, loss = {loss}')
    return loss, acc1


In [16]:
def main(args):


    max_test_acc1 = 0.


    utils.init_distributed_mode(args)
    print(args)
    output_dir = os.path.join(args.output_dir, f'{args.model}_b{args.batch_size}_lr{args.lr}_T{args.T}')

    if args.zero_init_residual:
        output_dir += '_zi'
    if args.weight_decay:
        output_dir += f'_wd{args.weight_decay}'

    output_dir += f'_coslr{args.cos_lr_T}'

    if args.adam:
        output_dir += '_adam'
    else:
        output_dir += '_sgd'

    if args.connect_f:
        output_dir += f'_cnf_{args.connect_f}'

    if output_dir:
        utils.mkdir(output_dir)
        if args.tb and utils.is_main_process():
            utils.mkdir(os.path.join(output_dir, '_logs'))


    device = torch.device(args.device)


    print("Creating model")

    model = _sew_resnet(zero_init_residual=args.zero_init_residual, connect_f=args.connect_f)
 
    model.to(device)
    if args.distributed and args.sync_bn:
        model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)

    criterion = SF.ce_temporal_loss()
    


    if args.adam:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    else:
        optimizer = torch.optim.SGD(
            model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)

    if args.amp:
        scaler = torch.amp.GradScaler('cuda')
    else:
        scaler = None


    #lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.cos_lr_T)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=1, eta_min=1e-6)



    model_without_ddp = model
    if args.distributed:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
        model_without_ddp = model.module

    if args.resume:
        checkpoint = torch.load(args.resume, map_location='cpu', weights_only=False)
        model_without_ddp.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])

        args.start_epoch = checkpoint['epoch'] + 1

        max_test_acc1 = checkpoint['max_test_acc1']

    if args.test_only:
        evaluate(model, criterion, val_loader, device=device, header='Test:')
        return


    print("Start training")
    start_time = time.time()
    for epoch in range(args.start_epoch, args.epochs):
        save_max = False
        if args.distributed:
            train_loader.set_epoch(epoch)
        train_loss, train_acc1 = train_one_epoch(model, criterion, optimizer, train_loader, device, epoch, args.print_freq, scaler)
       
        lr_scheduler.step()

        test_loss, test_acc1 = evaluate(model, criterion, val_loader, device=device, header='Test:')

        if max_test_acc1 < test_acc1:
            max_test_acc1 = test_acc1
            save_max = True



        if output_dir:

            checkpoint = {
                'model': model_without_ddp.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': lr_scheduler.state_dict(),
                'epoch': epoch,
                'args': args,
                'max_test_acc1': max_test_acc1
            }

            utils.save_on_master(
                checkpoint,
                os.path.join(output_dir, 'checkpoint_latest.pth'))
            save_flag = False

            if epoch % 64 == 0 or epoch == args.epochs - 1:
                save_flag = True

            elif args.cos_lr_T == 0:
                for item in args.lr_step_size:
                    if (epoch + 2) % item == 0:
                        save_flag = True
                        break

            if save_flag:
                utils.save_on_master(
                    checkpoint,
                    os.path.join(output_dir, f'checkpoint_{epoch}.pth'))

            if save_max:
                utils.save_on_master(
                    checkpoint,
                    os.path.join(output_dir, 'checkpoint_max_test_acc1.pth'))
        print(args)
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print(output_dir)

        print('Training time {}'.format(total_time_str), 'max_test_acc1', max_test_acc1)


# Model 

In [17]:
class Args:

    model = "TSEWResNet"
    T = 10                        # time steps
    zero_init_residual = False    
    connect_f = 'ADD'             
    batch_size = 128               
    epochs = 64                   
    lr = 1e-3
                          
    momentum = 0.9                # SGD momentum
    weight_decay = 1e-4           
    adam = True                  
    cos_lr_T = 64                 # T_max for CosineAnnealingLR scheduler
    lr_step_size = [30, 60]       # fallback step sizes if cos_lr_T is 0
    amp = True                    

    device = 'cuda'               
    distributed = False           
    gpu = 0                       
    sync_bn = False               

    resume = 'C:/Users/Veronika/Documents/NCALTECH_VS/ncaltech_snn/checkpoints/TSEWResNet_b128_lr0.001_T10_wd0.0001_coslr64_adam_cnf_ADD/checkpoint_max_test_acc1.pth'                   # path to a checkpoint .pth file
    test_only = False             
    tb = False                    
    print_freq = 50               
    output_dir = './checkpoints'  
    start_epoch = 0               

args = Args()

In [ ]:
main(args)

Not using distributed mode
Creating model


c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


Start training
Epoch: [33]  [ 0/47]  eta: 0:21:13  lr: 0.0004754661628362909  img/s: 0.3898893205454195  loss: 3.3136 (3.3136)  acc1: 33.5938 (33.5938)  time: 27.0916  data: 1.4433  max mem: 15975
Epoch: [33] Total time: 0:28:56
Test:  [ 0/13]  eta: 0:04:06  loss: 3.1210 (3.1210)  acc1: 35.9375 (35.9375)  time: 18.9995  data: 2.3359  max mem: 16006
Test: Total time: 0:03:25
 * Acc@1 = 35.75721153846154, loss = 3.1586652535658617
./checkpoints\TSEWResNet_b128_lr0.001_T10_wd0.0001_coslr64_adam_cnf_ADD
Training time 0:32:21 max_test_acc1 36.65865384615385
Epoch: [34]  [ 0/47]  eta: 0:27:36  lr: 0.0005  img/s: 0.3061456838937554  loss: 2.9846 (2.9846)  acc1: 39.0625 (39.0625)  time: 35.2439  data: 2.5797  max mem: 16006
Epoch: [34] Total time: 0:27:03
Test:  [ 0/13]  eta: 0:03:26  loss: 3.1330 (3.1330)  acc1: 39.8438 (39.8438)  time: 15.8891  data: 5.5082  max mem: 16006
Test: Total time: 0:03:00
 * Acc@1 = 36.23798076923077, loss = 3.1899920060084415
./checkpoints\TSEWResNet_b128_lr0.001_

In [18]:
model = _sew_resnet(zero_init_residual=args.zero_init_residual, connect_f=args.connect_f)

In [19]:
checkpoint_path = 'C:/Users/Veronika/Documents/NCALTECH_VS/ncaltech_snn/checkpoints/TSEWResNet_b128_lr0.001_T10_wd0.0001_coslr64_adam_cnf_ADD/checkpoint_max_test_acc1.pth'
checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)

c:\Users\Veronika\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_utils.py:106: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  untyped_storage = torch.UntypedStorage(self.size(), device=device)


In [22]:
model.load_state_dict(checkpoint['model'])
model.eval()
model.to('cuda')

TSEWResNet(
  (conv1): SeqToANNContainer(
    (0): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (bn1): SeqToANNContainer(
    (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (sn1): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (maxpool): SeqToANNContainer(
    (0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): SeqToANNContainer(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (sn1): IFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (conv2): SeqTo

In [23]:
evaluate(model, criterion= SF.ce_temporal_loss(), data_loader=test_loader, device=torch.device(args.device))

Test:  [0/6]  eta: 0:00:40  loss: 2.9846 (2.9846)  acc1: 42.9688 (42.9688)  time: 6.8036  data: 2.3132  max mem: 3991
Test: Total time: 0:00:19
 * Acc@1 = 35.9375, loss = 3.1910701592763266


(3.1910701592763266, 35.9375)

In [19]:
checkpoint_path = 'C:/Users/Veronika/Documents/NCALTECH_VS/T-ResNet/checkpoints/TSEWResNet_b128_lr0.001_T10_wd0.0001_coslr64_adam_cnf_ADD/checkpoint_latest.pth'
checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)

In [20]:
model.load_state_dict(checkpoint['model'])
model.eval()
model.to('cuda')

TSEWResNet(
  (conv1): SeqToANNContainer(
    (0): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  )
  (bn1): SeqToANNContainer(
    (0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (sn1): IFNode(
    v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
    (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
  )
  (maxpool): SeqToANNContainer(
    (0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): SeqToANNContainer(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (sn1): IFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, step_mode=m, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (conv2): SeqTo

In [21]:
evaluate(model, criterion= SF.ce_temporal_loss(), data_loader=test_loader, device=torch.device(args.device))

Test:  [0/6]  eta: 0:00:27  loss: 2.9582 (2.9582)  acc1: 40.6250 (40.6250)  time: 4.6470  data: 1.6931  max mem: 4024
Test: Total time: 0:00:14
 * Acc@1 = 36.588541666666664, loss = 3.1413673162460327


(3.1413673162460327, 36.588541666666664)